# Zadanie 1 — NumPy: tablice, selekcja i obliczenia wektorowe

**Czas:** 25–30 minut  
**Poziom:** podstawowy  
**Cel:** zamienić tabelę sześciu budynków na tablice NumPy i bez pętli obliczyć prostą prognozę oraz jej błąd.

**Zasady:** pracuj wyłącznie na danych zapisanych poniżej. Nie potrzebujesz internetu ani dodatkowego pliku. W miejscach oznaczonych `TODO` wpisz własny kod. Dane są syntetyczne, a wartości kWh służą tylko do ćwiczenia.


> **WERSJA DLA PROWADZĄCEGO — kompletne rozwiązanie.**
>
> Uruchom komórki od góry. Kod, wyniki kontrolne i interpretacje są częścią odpowiedzi. Dane są syntetyczne.


In [ ]:
import numpy as np

ids = np.array(["A01", "A02", "A03", "A04", "A05", "A06"])
areas = np.array([40., 65., 82., 55., 120., 73.])
years = np.array([2008, 2015, 1999, 2020, 1988, 2012])
kwh = np.array([180., 293., 369., 250., 620., 340.])

print("Liczba budynków:", len(ids))


## A. Zrozum kształt danych

1. Utwórz macierz `X` o **6 wierszach i 2 kolumnach**: pierwsza kolumna to metraż, druga to rok budowy. Użyj `np.column_stack`.
2. Wyświetl `X.shape`, `X.dtype`, cały pierwszy wiersz oraz całą kolumnę metrażu przez indeksowanie macierzy.
3. Sprawdź, czy pierwsza kolumna `X` zgadza się z tablicą `areas`.


In [ ]:
X = np.column_stack((areas, years))
print("Kształt X:", X.shape)
print("Typ elementów X:", X.dtype)
print("Pierwszy budynek:", X[0])
print("Kolumna metrażu:", X[:, 0])
print("Metraż zgodny z areas:", np.array_equal(X[:, 0], areas))


## B. Filtruj bez pętli

4. Utwórz maskę logiczną `large_mask` dla budynków o metrażu **większym niż 70 m²**.
5. Użyj maski, aby wypisać identyfikatory i zużycie kWh tych budynków.
6. Policz, ile budynków spełnia warunek. Użyj `np.sum` albo `.sum()`.


In [ ]:
large_mask = areas > 70
print("Maska:", large_mask)
print("Identyfikatory:", ids[large_mask])
print("Zużycie kWh:", kwh[large_mask])
print("Liczba budynków:", int(large_mask.sum()))


## C. Wektoryzacja i prosty model odniesienia

7. Przyjmij prognozę bazową **4,5 kWh na każdy m²**. W jednej operacji NumPy oblicz tablicę `baseline` dla wszystkich budynków.
8. Oblicz `error = kwh - baseline`, następnie `absolute_error` i średni błąd bezwzględny `mae`.
9. Oblicz `intensity = kwh / areas`. Który budynek ma największą intensywność zużycia? Podaj identyfikator i wynik zaokrąglony do dwóch miejsc.

**Interpretacja:** w komórce Markdown pod kodem napisz dwa zdania: co oznacza MAE i czy średni błąd ukrywa budynek, dla którego prognoza jest szczególnie słaba.


In [ ]:
baseline = areas * 4.5
error = kwh - baseline
absolute_error = np.abs(error)
mae = float(np.mean(absolute_error))
intensity = kwh / areas

highest_intensity_index = int(np.argmax(intensity))
worst_error_index = int(np.argmax(absolute_error))

print("Prognoza bazowa:", baseline)
print("Błędy ze znakiem:", error)
print("Błędy bezwzględne:", absolute_error)
print("MAE:", mae, "kWh")
print("Największa intensywność:", ids[highest_intensity_index],
      f"{intensity[highest_intensity_index]:.2f} kWh/m²")
print("Największy błąd:", ids[worst_error_index],
      absolute_error[worst_error_index], "kWh")


### Interpretacja prowadzącego

1. **MAE = 15,75 kWh** oznacza, że średni bezwzględny odstęp prognozy od pomiaru wynosi 15,75 kWh na budynek w tym zbiorze. Nie mówi nic o kierunku błędu.
2. Sama średnia ukrywa obiekt **A05**: prognoza dla niego myli się o **80 kWh**. Pozostałe pięć błędów jest dużo mniejszych, więc warto pokazać także rozkład błędów, a nie tylko MAE.


## Samokontrola

Uruchom tę komórkę **po wykonaniu wszystkich części**. Sprawdza kilka wyników, ale nie zastępuje interpretacji.


In [ ]:
assert X.shape == (6, 2)
assert np.array_equal(X[:, 0], areas)
assert int(np.sum(large_mask)) == 3
assert np.allclose(baseline, areas * 4.5)
assert np.isclose(mae, 15.75)
assert ids[np.argmax(intensity)] == "A05"
print("Kontrola liczbowa zakończona pomyślnie.")


## Wyzwanie dodatkowe

Policz MAE po usunięciu z oceny budynku o największym błędzie. Porównaj obie wartości i wyjaśnij, dlaczego raportowanie samej średniej może być mylące.


## Rozwiązanie wyzwania dodatkowego

Usuwamy z **oceny**, a nie z danych wejściowych, jeden rekord o największym błędzie. Porównujemy średnią błędów pozostałych pięciu obiektów. To analiza wrażliwości metryki, nie zalecenie usuwania niewygodnych przypadków z testu.


In [ ]:
mae_without_worst = float(np.mean(np.delete(absolute_error, worst_error_index)))
print("MAE dla wszystkich:", mae)
print("MAE bez obiektu", ids[worst_error_index], ":", mae_without_worst)
assert np.isclose(mae_without_worst, 2.9)


**Wniosek:** wynik spada z 15,75 do 2,90 kWh. Jeden budynek silnie wpływa na średnią. W rzeczywistej analizie należałoby sprawdzić, czy jest nietypowym, ale poprawnym przypadkiem, czy błędem danych. Raportujemy pełny wynik oraz analizę tego przypadku.
